# Explorative Analyse: Arbeitsmarkt Regensburg (Jobcenter) — Teil 3: Statistische Analyse

Baut auf den in [`01_Exploration.ipynb`](01_Exploration.ipynb) exportierten CSV-Dateien auf und
ergänzt [`02_Analyse_SGBII.ipynb`](02_Analyse_SGBII.ipynb) um vertiefte statistische Verfahren:
deskriptive Statistik im Detail, Korrelationsanalyse, Verteilungen, Inferenzstatistik
(Hypothesentest) und eine einfache lineare Regression.

**Wichtiger Hinweis zur Stichprobengröße:** Die Zeitreihe umfasst nur **18 Monatswerte**
(Januar 2025 – Juni 2026). Statistische Tests und Regressionen sind bei so wenigen Datenpunkten
nur eingeschränkt aussagekräftig — Ergebnisse werden entsprechend vorsichtig interpretiert.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from sklearn.linear_model import LinearRegression

PROCESSED_DIR = Path("..") / "Data" / "processed"

pd.set_option("display.max_columns", None)

kennzahlen = pd.read_csv(PROCESSED_DIR / "kennzahlen_sgb2.csv", index_col="period", parse_dates=["period"])
kennzahlen = kennzahlen.sort_index()

kern = ["Arbeitslose (Bestand)", "Langzeitarbeitslose", "Bedarfsgemeinschaften", "Personen in BG"]

kennzahlen[kern]

,Arbeitslose (Bestand),Langzeitarbeitslose,Bedarfsgemeinschaften,Personen in BG
period,,,,
2025-01-01,1470.0,502.0,2797.303381,5488.033274
2025-02-01,1382.0,502.0,2755.768665,5434.992911
2025-03-01,1414.0,513.0,2781.436728,5474.933615
2025-04-01,1457.0,526.0,2766.959069,5421.148834
2025-05-01,1438.0,558.0,2707.556182,5295.629265
2025-06-01,1509.0,551.0,2696.605759,5275.539536
2025-07-01,1475.0,561.0,2616.920054,5092.808598
2025-08-01,1587.0,585.0,2688.060659,5267.867812
2025-09-01,1585.0,592.0,2669.283813,5218.350099


## 1. Deskriptive Statistik im Detail

Über `describe()` und `mean`/`std` hinaus werden hier zusätzlich Median, Varianz, Interquartilsabstand (IQR) sowie der Variationskoeffizient (CV = Std.-Abw. / Mittelwert, in %) berechnet — Letzterer macht die Kennzahlen trotz unterschiedlicher Größenordnungen bezüglich ihrer relativen Streuung vergleichbar.

In [2]:
desc = pd.DataFrame({
    "Mittelwert": kennzahlen[kern].mean(),
    "Median": kennzahlen[kern].median(),
    "Std.-Abw.": kennzahlen[kern].std(),
    "Varianz": kennzahlen[kern].var(),
    "Min": kennzahlen[kern].min(),
    "Q1 (25%)": kennzahlen[kern].quantile(0.25),
    "Q3 (75%)": kennzahlen[kern].quantile(0.75),
    "Max": kennzahlen[kern].max(),
})
desc["IQR"] = desc["Q3 (75%)"] - desc["Q1 (25%)"]
desc["Variationskoeff. (%)"] = (desc["Std.-Abw."] / desc["Mittelwert"] * 100).round(1)
desc.round(2)

,Mittelwert,Median,Std.-Abw.,Varianz,Min,Q1 (25%),Q3 (75%),Max,IQR,Variationskoeff. (%)
Arbeitslose (Bestand),1504.00,1513.00,58.45,3416.12,1382.00,1467.00,1549.00,1587.00,82.00,3.9
Langzeitarbeitslose,602.22,599.00,70.84,5018.54,502.00,552.75,660.25,714.00,107.50,11.8
Bedarfsgemeinschaften,2691.72,2692.07,55.69,3100.91,2616.92,2643.59,2712.66,2797.30,69.07,2.1
Personen in BG,5247.39,5216.59,127.59,16280.01,5092.81,5143.69,5290.61,5488.03,146.92,2.4


**Beobachtung:** Die Langzeitarbeitslosen haben mit rund 11,8 % den mit Abstand höchsten Variationskoeffizienten aller vier Kernkennzahlen — ein quantitativer Beleg dafür, dass diese Kennzahl sich im Beobachtungszeitraum relativ am stärksten verändert hat, während Bedarfsgemeinschaften (2,1 %) und Personen in BG (2,4 %) vergleichsweise stabil sind.

## 2. Korrelationsanalyse

Die Pearson-Korrelationsmatrix zeigt lineare Zusammenhänge zwischen den vier Kernkennzahlen. Werte nahe +1/−1 bedeuten starken positiven/negativen linearen Zusammenhang, Werte nahe 0 keinen linearen Zusammenhang.

In [3]:
corr = kennzahlen[kern].corr()

fig = go.Figure(go.Heatmap(
    z=corr.values, x=corr.columns, y=corr.columns,
    zmin=-1, zmax=1, colorscale="RdBu", reversescale=True,
    text=corr.round(2).values, texttemplate="%{text}",
))
fig.update_layout(title="Korrelationsmatrix der Kernkennzahlen (Pearson)", height=500, width=650)
fig

**Auffälligster Wert:** Bedarfsgemeinschaften und Personen in BG korrelieren mit r ≈ 0,95 sehr stark positiv — plausibel, da die Personenzahl direkt aus der Zahl der Bedarfsgemeinschaften hervorgeht. Interessanter ist der ursprünglich vermutete Zusammenhang zwischen Langzeitarbeitslosen und Bedarfsgemeinschaften: er wird im folgenden Streudiagramm genauer geprüft.

In [4]:
r, p = stats.pearsonr(kennzahlen["Langzeitarbeitslose"], kennzahlen["Bedarfsgemeinschaften"])

fig = px.scatter(
    kennzahlen, x="Bedarfsgemeinschaften", y="Langzeitarbeitslose",
    trendline="ols", trendline_color_override="#dc2626",
)
fig.update_traces(marker=dict(size=9, color="#7c3aed"), selector=dict(mode="markers"))
fig.update_layout(
    title=f"Langzeitarbeitslose vs. Bedarfsgemeinschaften (r = {r:.2f}, p = {p:.2f})",
    xaxis_title="Bedarfsgemeinschaften", yaxis_title="Langzeitarbeitslose",
    height=450,
)
fig

**Ergebnis:** Mit r = −0,49 und p = 0,04 besteht ein **statistisch signifikanter, moderater
negativer Zusammenhang** zwischen Langzeitarbeitslosen und Bedarfsgemeinschaften: Je höher die
Langzeitarbeitslosigkeit, desto niedriger tendenziell die Zahl der Bedarfsgemeinschaften im
selben Monat — inhaltlich plausibel, da Bedarfsgemeinschaften über den Beobachtungszeitraum
insgesamt leicht sinken, während die Langzeitarbeitslosigkeit steigt. Bei den ursprünglichen
15 Monaten war dieser Zusammenhang mit r = −0,08 (p = 0,78) noch nicht nachweisbar — ein gutes
Beispiel dafür, wie empfindlich Korrelationsschätzungen bei kleinen Stichproben auf zusätzliche
Datenpunkte reagieren. Noch deutlicher zeigt sich der Zusammenhang zwischen Langzeitarbeitslosen
und dem allgemeinen Arbeitslosenbestand: r = 0,67 (p = 0,002) — jetzt klar signifikant, während
er mit 15 Monaten noch knapp unter der Signifikanzschwelle lag (r = 0,48, p = 0,07).

## 3. Verteilungen

Histogramm und Boxplot zeigen die Verteilung des Arbeitslosenbestands über die 18 Berichtsmonate — inkl. Median, Quartilen und möglicher Ausreißer.

In [5]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=kennzahlen["Arbeitslose (Bestand)"], nbinsx=8, marker_color="#2563eb"))
fig.update_layout(
    title="Verteilung: Arbeitslose (Bestand)",
    xaxis_title="Personen", yaxis_title="Anzahl Monate", height=400,
)
fig

In [6]:
fig = go.Figure()
fig.add_trace(go.Box(y=kennzahlen["Arbeitslose (Bestand)"], name="Arbeitslose (Bestand)", marker_color="#2563eb", boxmean=True))
fig.update_layout(title="Boxplot: Arbeitslose (Bestand)", yaxis_title="Personen", height=450, showlegend=False)
fig

Der Boxplot zeigt keine auffälligen Ausreißer (keine Punkte außerhalb der Whisker); die Verteilung ist leicht linksschief mit einem Median knapp über dem Mittelwert (siehe Tabelle in Kapitel 1).

## 4. Inferenzstatistik: Hypothesentest

**Fragestellung:** Unterscheidet sich die durchschnittliche Zahl der Langzeitarbeitslosen in der zweiten Hälfte des Beobachtungszeitraums signifikant von der ersten Hälfte?

**Hypothesen:** H₀: kein Unterschied der Mittelwerte. H₁: die zweite Hälfte hat einen höheren Mittelwert. Getestet wird mit einem zweiseitigen **Welch-t-Test** (ungleiche Varianzen werden nicht vorausgesetzt), Signifikanzniveau α = 0,05.

In [7]:
lza = kennzahlen["Langzeitarbeitslose"]
mitte = len(lza) // 2
erste_haelfte = lza.iloc[:mitte]
zweite_haelfte = lza.iloc[mitte:]

t_stat, p_wert = stats.ttest_ind(zweite_haelfte, erste_haelfte, equal_var=False)

diff = zweite_haelfte.mean() - erste_haelfte.mean()
se = np.sqrt(erste_haelfte.var(ddof=1) / len(erste_haelfte) + zweite_haelfte.var(ddof=1) / len(zweite_haelfte))
ci = (diff - 1.96 * se, diff + 1.96 * se)

print(f"1. Hälfte (n={len(erste_haelfte)}): Ø {erste_haelfte.mean():.1f} (± {erste_haelfte.std():.1f})")
print(f"2. Hälfte (n={len(zweite_haelfte)}): Ø {zweite_haelfte.mean():.1f} (± {zweite_haelfte.std():.1f})")
print(f"Mittelwertdifferenz: {diff:.1f} Personen (95%-CI: [{ci[0]:.1f}, {ci[1]:.1f}])")
print(f"Welch-t-Test: t = {t_stat:.2f}, p = {p_wert:.5f}")
print("Signifikant bei α=0.05:", "Ja" if p_wert < 0.05 else "Nein")

1. Hälfte (n=9): Ø 543.3 (± 34.1)
2. Hälfte (n=9): Ø 661.1 (± 41.2)
Mittelwertdifferenz: 117.8 Personen (95%-CI: [82.8, 152.7])
Welch-t-Test: t = 6.61, p = 0.00001
Signifikant bei α=0.05: Ja


**Interpretation:** Mit p ≈ 0,00001 (deutlich unter α = 0,05) wird H₀ verworfen — der
Anstieg der Langzeitarbeitslosigkeit von durchschnittlich 543 (erste Hälfte des Zeitraums) auf
661 Personen (zweite Hälfte) ist **statistisch hoch signifikant**, nicht durch zufällige
Schwankungen erklärbar. Das 95 %-Konfidenzintervall für die Differenz (ca. 83 bis 153
Personen) schließt die Null nicht ein, was die Signifikanz bestätigt.

## 5. Regressionsanalyse: Trend der Langzeitarbeitslosigkeit

Einfache lineare Regression der Langzeitarbeitslosen-Zahl über die Zeit (Monatsindex 0 = Januar 2025 bis 17 = Juni 2026), umgesetzt mit `sklearn.linear_model.LinearRegression`.

In [8]:
X = np.arange(len(lza)).reshape(-1, 1)
y = lza.values

modell = LinearRegression().fit(X, y)
r2 = modell.score(X, y)
steigung = modell.coef_[0]
achsenabschnitt = modell.intercept_

print(f"Regressionsgleichung: Langzeitarbeitslose = {achsenabschnitt:.1f} + {steigung:.2f} × Monatsindex")
print(f"Steigung: +{steigung:.2f} Personen pro Monat")
print(f"Bestimmtheitsmaß R²: {r2:.3f}")

# Prognose fuer die naechsten 3 Monate (Juli-September 2026)
X_prognose = np.arange(len(lza), len(lza) + 3).reshape(-1, 1)
y_prognose = modell.predict(X_prognose)
monate_prognose = pd.date_range(kennzahlen.index[-1], periods=4, freq="MS")[1:]
for monat, wert in zip(monate_prognose, y_prognose):
    print(f"Prognose {monat:%b %Y}: {wert:.0f} Personen")

Regressionsgleichung: Langzeitarbeitslose = 490.4 + 13.16 × Monatsindex
Steigung: +13.16 Personen pro Monat
Bestimmtheitsmaß R²: 0.983
Prognose Jul 2026: 727 Personen
Prognose Aug 2026: 740 Personen
Prognose Sep 2026: 754 Personen


In [9]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=kennzahlen.index, y=lza.values, mode="markers", name="Beobachtet", marker=dict(color="#7c3aed", size=8)))

y_fit = modell.predict(X)
fig.add_trace(go.Scatter(x=kennzahlen.index, y=y_fit, mode="lines", name="Regressionsgerade", line=dict(color="#dc2626", width=2)))

fig.add_trace(go.Scatter(x=monate_prognose, y=y_prognose, mode="markers+lines", name="Prognose (3 Monate)",
                          marker=dict(color="#dc2626", symbol="diamond", size=9), line=dict(color="#dc2626", dash="dash")))

fig.update_layout(
    title=f"Langzeitarbeitslose: linearer Trend und Prognose (R² = {r2:.2f})",
    xaxis_title="Monat", yaxis_title="Personen", height=480,
    legend=dict(orientation="h", y=1.12),
)
fig

**Interpretation:** Mit **R² = 0,983** erklärt die lineare Zeit-Trendlinie rund 98,3 % der
Schwankung der Langzeitarbeitslosen-Zahl — ein außergewöhnlich guter linearer Fit, sogar noch
etwas besser als mit den ursprünglichen 15 Monaten (R² = 0,976). Die Steigung von
**+13,2 Personen pro Monat** entspricht der bereits in Kapitel 3.3/3.4 von
`02_Analyse_SGBII.ipynb` beschriebenen Entwicklung. Die Prognose für die folgenden drei
Monate (Juli–September 2026) liegt bei rund 727, 740 und 754 Personen.

**Einschränkung:** Die Prognose extrapoliert einen linearen Trend aus nur 18 Datenpunkten in
die Zukunft. Sie unterstellt, dass sich der bisherige Trend unverändert fortsetzt, und
berücksichtigt keine saisonalen Effekte, Strukturbrüche oder externe Einflüsse (z. B.
konjunkturelle Entwicklung, Gesetzesänderungen). Die Prognose ist daher als grobe
Orientierung zu verstehen, nicht als verlässliche Vorhersage.

## 6. Zusammenfassung

- **Deskriptive Statistik:** Langzeitarbeitslose weisen mit einem Variationskoeffizienten von
  rund 11,8 % die stärkste relative Veränderung aller vier Kernkennzahlen auf;
  Bedarfsgemeinschaften (2,1 %) und Personen in BG (2,4 %) sind am stabilsten.
- **Korrelation:** Mit den jetzt 18 Monaten zeigt sich ein statistisch signifikanter,
  moderater negativer Zusammenhang zwischen Langzeitarbeitslosen und Bedarfsgemeinschaften
  (r = −0,49, p = 0,04) sowie ein deutlicher positiver Zusammenhang zwischen
  Langzeitarbeitslosen und dem allgemeinen Arbeitslosenbestand (r = 0,67, p = 0,002) — beide
  Zusammenhänge waren mit den ursprünglichen 15 Monaten noch nicht signifikant nachweisbar.
  Bedarfsgemeinschaften und Personen in BG korrelieren weiterhin sehr stark (r ≈ 0,95).
- **Hypothesentest:** Der Anstieg der Langzeitarbeitslosigkeit zwischen erster (Ø 543) und
  zweiter (Ø 661) Hälfte des Beobachtungszeitraums ist statistisch hoch signifikant
  (p ≈ 0,00001).
- **Regression:** Ein linearer Zeittrend erklärt mit R² = 0,983 nahezu die gesamte
  Schwankung der Langzeitarbeitslosen-Zahl; die Prognose für die kommenden drei Monate liegt
  bei weiterem Anstieg auf rund 727–754 Personen — allerdings mit den oben genannten
  Einschränkungen einer einfachen linearen Extrapolation.

Diese statistischen Befunde untermauern quantitativ, was in `02_Analyse_SGBII.ipynb` bereits
visuell erkennbar war: Die Verschärfung der Langzeitarbeitslosigkeit ist der mit Abstand
robusteste und am stärksten abgesicherte Trend im gesamten Datensatz. Bemerkenswert ist
zudem, dass die zusätzlichen drei Monate Rohdaten zwei zuvor statistisch nicht nachweisbare
Zusammenhänge (Korrelation mit Bedarfsgemeinschaften und mit dem Arbeitslosenbestand) erst
sichtbar gemacht haben — ein anschauliches Beispiel für die Bedeutung der Stichprobengröße
bei Signifikanztests.